# 02 · Thinking in N dimensions

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/02-thinking-in-n-dimensions.ipynb)

*Part II · group · 20 min*

> 🇪🇸 **Pensar en N dimensiones** — Discutir qué significa cada eje y por qué un eje de lote difiere de un eje temporal.

Argue about what each axis means, and why a batch axis differs from a time axis.

## What you will be able to do

- Say what a new axis *counts*, rather than saying "we add a dimension".
- Explain why a batch axis and a time axis behave differently despite identical shapes.
- Propose two ways to batch videos of different lengths, and say what each loses or invents.
- Map experimental choices onto axes of a real microscopy tensor.

## Setup

Run this first. It installs and imports everything this notebook needs, and nothing else.

> 🇪🇸 Ejecuta esto primero: instala e importa todo lo que este cuaderno necesita.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)

## This one is a discussion, not an exercise

> 🇪🇸 Este bloque es de discusión en grupo. Diez minutos de debate y luego
> puesta en común. **Sin código** al principio: dibuja en la pizarra compartida.

Go to your breakout channel. **No code at first.** Sketch on the shared board.
10 minutes discussion, then share-back. The code cells further down are for the
share-back — leave them alone until then.

> A grayscale image is a matrix: two axes, height and width. Almost nothing in
> machine learning is a single grayscale image. Each thing you add — colour,
> many examples, time — adds an axis, and each axis means something different.
> Your task is to argue about which axis goes where, and why.

### The five questions

1. Start from a grayscale image `(H, W)`. What is the shape of **(a)** one
   colour image, **(b)** a batch of colour images, **(c)** one video, **(d)** a
   batch of videos? For each step, say what the new axis *counts*.
   **Do not say "we add a dimension."**
2. A batch axis and a time axis both look like ordinary integer indices in code.
   What is different about their **meaning**? Think about what happens if you
   shuffle the order along each one.
3. Batching requires every example to have the same shape, but real videos have
   different numbers of frames. Propose two ways to build one batched tensor
   from videos of different lengths. What does each one lose or invent?
4. Photographing the same dish of cells every 10 minutes for 48 hours gives a
   tensor with the same shape as a video. Which experimental choice maps to
   which axis: frame interval → ? field of view → ? number of dishes → ?
5. Is there a mathematical limit on how many axes a tensor can have? If not,
   what actually limits you when you are writing the code?

## Exercise 1 — write down your group's answer to question 1

> 🇪🇸 Escribe la respuesta de tu grupo a la pregunta 1.

Do this *after* you have argued about it, not instead of arguing about it.

In [ ]:
# TODO 1: Create one array for each of the five stages, using the shapes your
#         group agreed on. Print each shape with a comment saying what the NEW
#         axis counts at that step.

gray_image      = np.zeros((28, 28))
color_image     = ...   # + colour
batch_of_images = ...   # + many examples
video           = ...   # + ordered time
batch_of_videos = ...   # + many examples of ordered time

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
gray_image      = np.zeros((28, 28))            # (H, W)
color_image     = np.zeros((28, 28, 3))         # (H, W, C)      + colour
batch_of_images = np.zeros((32, 28, 28, 3))     # (N, H, W, C)   + many examples
video           = np.zeros((16, 28, 28, 3))     # (T, H, W, C)   + ordered time
batch_of_videos = np.zeros((8, 16, 28, 28, 3))  # (N, T, H, W, C)

for name, a in [("gray", gray_image), ("colour", color_image),
                ("batch", batch_of_images), ("video", video),
                ("batch of videos", batch_of_videos)]:
    print(f"{name:16s} {a.shape}  order {a.ndim}")

## Exercise 2 — question 2, in code

> 🇪🇸 La pregunta 2, demostrada con código.

`batch_of_images` and `video` have the same *kind* of shape tuple. Question 2
claims they behave completely differently. Show it.

In [ ]:
# TODO 2: Shuffle axis 0 of `batch_of_images` and argue why nothing is lost.
#         Then shuffle axis 0 of `video` and argue what exactly was destroyed.
#         Hint: put something recognisable along the axis first, so you can see
#         the damage — np.arange broadcast into each frame works well.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
# Label each position along axis 0 so the shuffle is visible.
batch = np.arange(8)[:, None, None] * np.ones((8, 4, 4))
video = np.arange(8)[:, None, None] * np.ones((8, 4, 4))

perm = rng.permutation(8)
print(batch[perm][:, 0, 0])   # e.g. [3. 0. 6. ...] — a different order
print(video[perm][:, 0, 0])   # the same numbers, and that is the problem

# The arrays are identical, and so is the operation. The DIFFERENCE IS MEANING:
#   batch — examples are independent, order carries no information.
#           Shuffling is harmless; every training loop does it on purpose.
#   video — order IS the information. Shuffled frames are no longer a video,
#           and nothing in the shape, dtype or size records that damage.

## Share-back

> 🇪🇸 Puesta en común.

```python
gray_image      = np.zeros((28, 28))            # (H, W)
color_image     = np.zeros((28, 28, 3))         # (H, W, C)      + colour
batch_of_images = np.zeros((32, 28, 28, 3))     # (N, H, W, C)   + many examples
video           = np.zeros((16, 28, 28, 3))     # (T, H, W, C)   + ordered time
batch_of_videos = np.zeros((8, 16, 28, 28, 3))  # (N, T, H, W, C)
```

The key idea is **question 2**. `batch_of_images` and `video` have the same
*kind* of shape tuple, but shuffling axis 0 is harmless for a batch — examples
are independent, order carries no information — and destroys a video, where
order **is** the information.

Chapter 2's notation has no concept of "order matters between elements." That is
genuinely new today.

Each step below adds exactly one axis — the one highlighted in red — onto the
tensor above it.

> 🇪🇸 Cada fila añade exactamente un eje — el resaltado en rojo — sobre el de
> arriba.

In [ ]:
import matplotlib.pyplot as plt

# name, axis labels in order, the one that's new at this step
stages = [
    ("gray_image",      ["H", "W"],              None),
    ("color_image",     ["H", "W", "C"],          "C"),
    ("batch_of_images", ["N", "H", "W", "C"],     "N"),
    ("video",           ["T", "H", "W", "C"],     "T"),
    ("batch_of_videos", ["N", "T", "H", "W", "C"], "N"),
]

fig, ax = plt.subplots(figsize=(7.5, 3.5))
for row, (name, axes, new) in enumerate(stages):
    y = len(stages) - row
    for col, axis in enumerate(axes):
        color = "#C44E52" if axis == new else "#4C72B0"
        ax.add_patch(plt.Rectangle((col, y - 0.4), 0.9, 0.8,
                                    facecolor=color, edgecolor="black"))
        ax.text(col + 0.45, y, axis, ha="center", va="center",
                color="white", fontweight="bold")
    ax.text(-0.2, y, name, ha="right", va="center", fontsize=9)

ax.set_xlim(-3.2, 5)
ax.set_ylim(0.3, len(stages) + 0.7)
ax.axis("off")
ax.set_title("Same batch/time-shaped tuple, different axis added each step")
plt.tight_layout()
plt.show()

### The other four, briefly

- **Q3** — pad every video to the longest and carry a mask (invents frames that
  were never recorded, and you must remember to ignore them), or sample a fixed
  number of frames from each (loses everything you did not sample). Take-home B
  in section 11 builds the mask.
- **Q4** — frame interval → the time axis; field of view → the height and width
  axes; number of dishes → a batch axis. Same shape as a video, completely
  different experiment.
- **Q5** — no mathematical limit. What limits you is memory, which grows as the
  product of the shape, and your own ability to remember what each axis means,
  which is why sections 03 and 04 exist.

---

## Done with this section

Next up: **03 · Indexing and broadcasting real data** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/03-indexing-and-broadcasting.ipynb).

[← Back to the workshop site](https://project-delphi.github.io/tensors-workshop/) · [All notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)